# 🧠 XAI — Explicabilidad del Modelo

Interpretación del modelo entrenado usando:
- **Feature importances** (intrínsecas del modelo)
- **SHAP** (explicaciones agnósticas)

Objetivo: entender *por qué* el modelo toma las decisiones que toma.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import joblib

from src.modules.iris_classifier.data_processing.loader import load_splits

%matplotlib inline

## 1. Cargar modelo y datos

In [2]:
model = joblib.load("../data/models/artifacts/iris_classifier/model.pkl")
X_train, X_test, y_train, y_test = load_splits()

FEATURE_NAMES = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
TARGET_NAMES = ["setosa", "versicolor", "virginica"]

## 2. Feature Importances (intrínsecas)

In [3]:
# Solo disponible para modelos basados en árboles
if hasattr(model, "feature_importances_"):
    importances = pd.Series(model.feature_importances_, index=FEATURE_NAMES)
    importances = importances.sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(8, 4))
    importances.plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title("Feature Importances (modelo entrenado)")
    ax.set_xlabel("Importancia")
    plt.tight_layout()
    plt.show()
else:
    print("El modelo no tiene feature_importances_. Se usará solo SHAP.")

El modelo no tiene feature_importances_. Se usará solo SHAP.


## 3. SHAP — Valores Shapley

SHAP asigna a cada feature una contribución a la predicción de cada muestra.
Se usa `KernelExplainer` ya que el modelo es un SVM (no basado en árboles).

In [ ]:
# KernelExplainer es agnóstico al modelo (funciona con SVM, etc.)
# Usamos un resumen del training set como background para eficiencia
background = shap.kmeans(X_train, 50)
explainer = shap.KernelExplainer(model.predict_proba, background)
shap_values = explainer.shap_values(X_test)

# KernelExplainer puede devolver un ndarray 3D (n_samples, n_features, n_classes)
# Convertimos a lista de arrays [clase_0, clase_1, ...] para compatibilidad con los plots
if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    shap_values = [shap_values[:, :, i] for i in range(shap_values.shape[2])]

Exception in thread Thread-7 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\javie\anaconda3\envs\ia_demo\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "c:\Users\javie\anaconda3\envs\ia_demo\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\javie\anaconda3\envs\ia_demo\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xa2 in position 116: invalid start byte


  0%|          | 0/30 [00:00<?, ?it/s]

## 4. SHAP — Summary Plot (global)

In [6]:
# Summary plot para cada clase
for i, name in enumerate(TARGET_NAMES):
    print(f"\n--- Clase: {name} ---")
    shap.summary_plot(shap_values[i], X_test, feature_names=FEATURE_NAMES, show=True)


--- Clase: setosa ---


AssertionError: The shape of the shap_values matrix does not match the shape of the provided data matrix.

## 5. SHAP — Bar Plot (importancia media)

In [ ]:
shap.summary_plot(
    shap_values, X_test, feature_names=FEATURE_NAMES,
    plot_type="bar", class_names=TARGET_NAMES
)

## 6. SHAP — Explicación de una predicción individual

In [ ]:
# Seleccionar una muestra de ejemplo
sample_idx = 0
sample = X_test.iloc[[sample_idx]]
pred = model.predict(sample)[0]

print(f"Muestra: {sample.values[0]}")
print(f"Predicción: {TARGET_NAMES[pred]}")
print()

# Waterfall plot para la clase predicha
shap.initjs()

# KernelExplainer: expected_value es una lista de arrays (uno por clase)
base_value = explainer.expected_value[pred]

explanation = shap.Explanation(
    values=shap_values[pred][sample_idx],
    base_values=base_value,
    data=X_test.iloc[sample_idx].values,
    feature_names=FEATURE_NAMES,
)
shap.waterfall_plot(explanation)

## 7. SHAP — Dependence Plot

In [ ]:
# Dependencia de petal_length (típicamente la feature más importante)
shap.dependence_plot(
    "petal_length", shap_values[2], X_test,
    feature_names=FEATURE_NAMES, interaction_index="petal_width"
)

## 8. Conclusiones

- Completar tras ejecutar el notebook.
- Verificar que `petal_length` y `petal_width` dominan las decisiones.
- Analizar cómo las features contribuyen de forma diferente para cada clase.